In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from datasets import load_dataset

ds = load_dataset("victor-buhl/COLREGS_ALPACA_SHORT")

print(ds)

example = ds["train"][0]

print("INSTRUCTION:", example["instruction"])
print("INPUT:", example["input"])
print("OUTPUT:", example["output"])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 980
    })
})
INSTRUCTION: INLAND ONLY You are navigating in a narrow channel and must remain in the channel for safe operation. Another vessel is crossing the channel ahead of you from your starboard and you doubt whether your vessel will pass safely. Which  statement is TRUE?
INPUT: Choice A: You must stop your vessel, since the other vessel is the stand-on. Choice B: You must sound one short blast of the whistle and turn to starboard. Choice C: You must sound the danger signal. Choice D: You must stop your engines and you may sound the danger  signal.
OUTPUT: Choice C: You must sound the danger signal.


In [ ]:
print(type(ds))
ds = ds['train']

outputs = ds["output"]
instructions = ds["instruction"]

lengths = [len(o.split()) for o in outputs]
print("Средняя длина output (слов):", sum(lengths)/len(lengths))
print("Макс длина:", max(lengths))
print("Мин длина:", min(lengths))

print("Всего строк:", len(instructions))
print("Уникальных instruction:", len(set(instructions)))

<class 'datasets.dataset_dict.DatasetDict'>
Средняя длина output (слов): 8.844897959183674
Макс длина: 37
Мин длина: 3
Всего строк: 980
Уникальных instruction: 922


In [ ]:
import json

seen = set()
cleaned = []

for ex in ds:
    instr = ex["instruction"].strip()
    inp = ex["input"].strip() if ex["input"] else ""
    out = ex["output"].strip()

    full_instruction = instr + ("\n" + inp if inp else "")

    if not full_instruction or not out:
        continue
    if full_instruction in seen:
        continue
    if len(out.split()) < 2:
        continue

    seen.add(full_instruction)
    cleaned.append({"instruction": full_instruction, "response": out})

print("Итоговое число примеров после очистки:", len(cleaned))

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for row in cleaned:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Готово, сохранено в dataset.jsonl")

Итоговое число примеров после очистки: 979
Готово, сохранено в dataset.jsonl


In [ ]:
with open("dataset.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(line) for line in f]

print("Всего строк:", len(lines))
print("\nПример 1:")
print(lines[0])
print("\nПример 500:")
print(lines[500])

Всего строк: 979

Пример 1:
{'instruction': 'INLAND ONLY You are navigating in a narrow channel and must remain in the channel for safe operation. Another vessel is crossing the channel ahead of you from your starboard and you doubt whether your vessel will pass safely. Which  statement is TRUE?\nChoice A: You must stop your vessel, since the other vessel is the stand-on. Choice B: You must sound one short blast of the whistle and turn to starboard. Choice C: You must sound the danger signal. Choice D: You must stop your engines and you may sound the danger  signal.', 'response': 'Choice C: You must sound the danger signal.'}

Пример 500:
{'instruction': 'BOTH INTERNATIONAL & INLAND Concerning the identification §signal for a pilot vessel, in fog, which statement is TRUE?\nChoice A: When at anchor, the pilot vessel is only required to sound anchor signals. Choice B: The identification signal must be sounded any time the pilot vessel is underway. Choice C: The pilot vessel may only soun

In [ ]:
import random

with open("dataset.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(line) for line in f]

print("Всего строк:", len(lines))

random.seed(42)
random.shuffle(lines)

eval_size = 30
eval_set = lines[:eval_size]
train_set = lines[eval_size:]

print("Train:", len(train_set))
print("Eval:", len(eval_set))

with open("train.jsonl", "w", encoding="utf-8") as f:
    for row in train_set:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open("eval.jsonl", "w", encoding="utf-8") as f:
    for row in eval_set:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Готово: train.jsonl и eval.jsonl сохранены")

Всего строк: 979
Train: 949
Eval: 30
Готово: train.jsonl и eval.jsonl сохранены


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Модель загружена")
print(model)

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Модель загружена
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (no

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("Модель подготовлена к k-bit тренировке")

Модель подготовлена к k-bit тренировке


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(train_dataset)
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 949
})
{'instruction': 'BOTH INTERNATIONAL & INLAND You see a vessel displaying the§code flag "LIMA" below which is a red ball. The vessel is§_____.\nChoice A: trolling Choice B: getting ready to receive aircraft Choice C: aground Choice D: in distress', 'response': 'Choice D: in distress'}


In [ ]:
def formatting_func(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return text

sample_text = formatting_func(train_dataset[0])
print(sample_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
BOTH INTERNATIONAL & INLAND You see a vessel displaying the§code flag "LIMA" below which is a red ball. The vessel is§_____.
Choice A: trolling Choice B: getting ready to receive aircraft Choice C: aground Choice D: in distress<|im_end|>
<|im_start|>assistant
Choice D: in distress<|im_end|>



In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-colregs-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    report_to="none",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
)

print("Trainer готов")

Applying formatting function to train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/949 [00:00<?, ? examples/s]

Trainer готов


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alibek-musabek (alibek-musabek-aitu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen-colregs-lora",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=3,
    learning_rate=2e-4,

    logging_steps=10,
    save_strategy="epoch",

    bf16=True,
    gradient_checkpointing=True,

    report_to="wandb",
    run_name="qwen2.5-7b-colregs-qlora",

    max_length=512,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
)

print("Trainer готов")

Trainer готов


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,1.715343
20,0.867807
30,0.829966
40,0.755015
50,0.714988
60,0.734935
70,0.728885
80,0.712300
90,0.713067
100,0.706477


TrainOutput(global_step=357, training_loss=0.521454137914321, metrics={'train_runtime': 11444.0151, 'train_samples_per_second': 0.249, 'train_steps_per_second': 0.031, 'total_flos': 1.3761300292411392e+16, 'train_loss': 0.521454137914321, 'entropy': 0.2886925536506581, 'num_tokens': 322536.0, 'mean_token_accuracy': 0.9248308370698173, 'epoch': 3.0})

In [ ]:
trainer.save_model("./qwen-colregs-lora-final")
tokenizer.save_pretrained("./qwen-colregs-lora-final")

('./qwen-colregs-lora-final/tokenizer_config.json',
 './qwen-colregs-lora-final/chat_template.jinja',
 './qwen-colregs-lora-final/tokenizer.json')

In [ ]:
wandb.finish()

train/entropy,█▅▄▄▄▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▃▁▃▂▂▂▃▂▂▂▂▂▅▃▅▄▃▄▆▅▃▅▅▃█▆▆▄▄▅▆▄▆▇▆
train/learning_rate,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,█▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇████████████
train/num_tokens,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
total_flos,1.3761300292411392e+16
train/entropy,0.28869
train/epoch,3


In [1]:
!git clone https://github.com/aliblackk/colregs-llm-qlora.git

Cloning into 'colregs-llm-qlora'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 23 (delta 0), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (23/23), 62.80 MiB | 10.20 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Updating files: 100% (15/15), done.


In [3]:
!pip install -q transformers peft bitsandbytes accelerate rouge-score pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.6 MB/s eta 0:00:00


In [4]:
import os
import json
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel
from rouge_score import rouge_scorer

In [6]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = "/content/colregs-llm-qlora/adapter/"
EVAL_FILE = "/content/colregs-llm-qlora/data/eval.jsonl"

NUM_EXAMPLES = 20
MAX_NEW_TOKENS = 256


with open(EVAL_FILE, "r", encoding="utf-8") as f:
    eval_data = [json.loads(line) for line in f]

# Берём максимум 20 примеров
eval_data = eval_data[:NUM_EXAMPLES]


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model.eval()

finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

finetuned_model.eval()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [7]:
def generate_answer(model, instruction, input_text=""):

    if input_text and input_text.strip():
        user_content = (
            instruction.strip()
            + "\n\n"
            + input_text.strip()
        )
    else:
        user_content = instruction.strip()

    messages = [
        {
            "role": "user",
            "content": user_content
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Берём только сгенерированную часть
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [8]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

results = []

base_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
finetuned_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for i, example in enumerate(eval_data, 1):

    instruction = example.get("instruction", "")
    reference = example.get("response", "")
    input_text = ""

    base_answer = generate_answer(
        base_model,
        instruction,
        input_text
    )

    finetuned_answer = generate_answer(
        finetuned_model,
        instruction,
        input_text
    )

    base_rouge = scorer.score(reference, base_answer)
    finetuned_rouge = scorer.score(reference, finetuned_answer)

    for metric in ["rouge1", "rouge2", "rougeL"]:
        base_scores[metric].append(
            base_rouge[metric].fmeasure
        )
        finetuned_scores[metric].append(
            finetuned_rouge[metric].fmeasure
        )

    print(
        f"Example {i}/{len(eval_data)} "
        f"Base ROUGE-L: {base_rouge['rougeL'].fmeasure:.4f} "
        f"Fine-tuned ROUGE-L: {finetuned_rouge['rougeL'].fmeasure:.4f}"
    )

    results.append({
        "example": i,
        "instruction": instruction,
        "reference": reference,
        "base_answer": base_answer,
        "finetuned_answer": finetuned_answer,
        "base_rouge1": base_rouge["rouge1"].fmeasure,
        "base_rouge2": base_rouge["rouge2"].fmeasure,
        "base_rougeL": base_rouge["rougeL"].fmeasure,
        "finetuned_rouge1": finetuned_rouge["rouge1"].fmeasure,
        "finetuned_rouge2": finetuned_rouge["rouge2"].fmeasure,
        "finetuned_rougeL": finetuned_rouge["rougeL"].fmeasure,
    })

summary = {
    "Base Model": {
        "ROUGE-1": sum(base_scores["rouge1"]) / len(base_scores["rouge1"]),
        "ROUGE-2": sum(base_scores["rouge2"]) / len(base_scores["rouge2"]),
        "ROUGE-L": sum(base_scores["rougeL"]) / len(base_scores["rougeL"]),
    },
    "Fine-tuned Model": {
        "ROUGE-1": sum(finetuned_scores["rouge1"]) / len(finetuned_scores["rouge1"]),
        "ROUGE-2": sum(finetuned_scores["rouge2"]) / len(finetuned_scores["rouge2"]),
        "ROUGE-L": sum(finetuned_scores["rougeL"]) / len(finetuned_scores["rougeL"]),
    }
}

summary_df = pd.DataFrame(summary).T

improvement = {
    metric:
        summary["Fine-tuned Model"][metric]
        - summary["Base Model"][metric]
    for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
}

improvement_df = pd.DataFrame(
    [improvement],
    index=["Improvement"]
)

print("\nAverage scores:")
print(summary_df.round(4))

print("\nImprovement:")
print(improvement_df.round(4))

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

summary_df.to_csv(
    "evaluation_summary.csv"
)

Example 1/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 2/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 3/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 4/20 Base ROUGE-L: 0.5333 Fine-tuned ROUGE-L: 0.5333
Example 5/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 6/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 7/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 8/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 9/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 10/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 11/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 12/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 13/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 14/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 15/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 16/20 Base ROUGE-L: 1.0000 Fine-tuned ROUGE-L: 1.0000
Example 17/20 Bas